# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

# Voigt function

In [ ]:
from skewerjax.optical_depth.voigt import log10_voigt

### Numerical errors and comparison between implementations

In [ ]:
xx = jnp.linspace(-9, 9, 2**13)

sigma_low = 1e-3
sigma_high = 1.0
gamma_low = 1e-3
gamma_high = 1.0
sigma = jnp.array([sigma_low, sigma_high, sigma_low, sigma_high])
gamma = jnp.array([gamma_low, gamma_low, gamma_high, gamma_high])

Voigt_dict = {}

Voigt_dict['scipy_wofz'] = log10_voigt(xx, sigma, gamma, mode="scipy_wofz")

voigt_convolution_batched = jit(
    vmap(lambda s, g: log10_voigt(xx, s, g, mode="convolution", oversample=4), in_axes=(0, 0))
)
Voigt_dict["JAX_convolution"] = voigt_convolution_batched(sigma, gamma)

voigt_exojax_batched = jit(
    vmap(lambda s, g: log10_voigt(xx, s, g, mode="ExoJAX"), in_axes=(0, 0))
)
Voigt_dict["exojax"] = voigt_exojax_batched(sigma, gamma)

In [ ]:
P = len(sigma)
colors = np.array(plt.cm.get_cmap("tab10").colors[:P])

fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))

for ii in range(P):
    ax.plot(xx, 10**Voigt_dict['scipy_wofz'][ii], color=colors[ii], linestyle="-", linewidth=1.0)
    ax.plot(xx, 10**Voigt_dict['JAX_convolution'][ii], color=colors[ii], linestyle="--", linewidth=2.)
    ax.plot(xx, 10**Voigt_dict['exojax'][ii], color=colors[ii], linestyle=":", linewidth=3.5)

for ii in range(P):
    ax_res.plot(
        xx, 10**Voigt_dict['JAX_convolution'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle="--", linewidth=1.0
    )
    ax_res.plot(
        xx, 10**Voigt_dict['exojax'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle=":", linewidth=2.0
    )

method_handles = [
    Line2D([0], [0], color="k", lw=1,  ls="-",  label="Scipy wofz"),
    Line2D([0], [0], color="k", lw=2,  ls="--", label="JAX convolution"),
    Line2D([0], [0], color="k", lw=3.5,  ls=":",  label="exojax"),
]
leg1 = ax.legend(handles=method_handles, loc="upper right", fontsize=16)
ax.add_artist(leg1)

param_handles = []
for ii in range(P):
    lbl = rf"$\sigma={sigma[ii]:g},\ \gamma={gamma[ii]:g}$"
    param_handles.append(Line2D([0], [0], color=colors[ii], lw=6, label=lbl))

leg2 = ax.legend(handles=param_handles, loc="upper left", fontsize=16)
ax.add_artist(leg2)

ax.set_title("Voigt profiles")
ax.set_ylabel(r"$V(x;\ \sigma,\ \gamma)$")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1e3)

ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res.set_xlabel("x")        # only bottom panel gets the x-label
ax_res.set_ylabel("residual")
ax_res.set_yscale("symlog", linthresh=1e-6)

major_ticks = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res.set_yticks(major_ticks)

ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

plt.subplots_adjust(hspace=0.05)
fig.tight_layout()
plt.show()

### Autodiff

In [ ]:
xx = jnp.linspace(-9.0, 9.0, 2**12)  # frequency (or velocity) offset grid

sigma0 = 1.0
gamma0 = 0.001

In [ ]:
def voigt_scalar(x, sigma, gamma):
    return 10**log10_voigt(jnp.atleast_1d(x), sigma, gamma, mode="ExoJAX")[0]

# Derivative wrt x (ν)
dVoigt_dx = vmap(grad(voigt_scalar, argnums=0), in_axes=(0, None, None))

# Derivative wrt sigma (Gaussian width)
dVoigt_dsigma = vmap(grad(voigt_scalar, argnums=1), in_axes=(0, None, None))

# Derivative wrt sigma (Lorentzian width)
dVoigt_dgamma = vmap(grad(voigt_scalar, argnums=2), in_axes=(0, None, None))

dV_dx = dVoigt_dx(xx, sigma0, gamma0)
dV_dsigma = dVoigt_dsigma(xx, sigma0, gamma0)
dV_dgamma = dVoigt_dgamma(xx, sigma0, gamma0)

In [ ]:
delta_sigma = 0.2
sigma = jnp.linspace(sigma0 - delta_sigma, sigma0 + delta_sigma, 5)
sigma_ref_idx = len(sigma) // 2
sigma_ref = sigma[sigma_ref_idx]

V_sigma = vmap(lambda s: 10**log10_voigt(xx, s, gamma0), in_axes=(0,))(sigma)
V_sigma_ref = 10**log10_voigt(xx, sigma_ref, gamma0)

delta_gamma = 0.0009
gamma = jnp.linspace(gamma0 - delta_gamma, gamma0 + delta_gamma, 5)
gamma_ref_idx = len(gamma) // 2
gamma_ref = gamma[gamma_ref_idx]

V_gamma = vmap(lambda g: 10**log10_voigt(xx, sigma0, g), in_axes=(0,))(gamma)
V_gamma_ref = 10**log10_voigt(xx, sigma0, gamma_ref)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex='col', gridspec_kw=dict(height_ratios=[2, 1]))

ax_main_s, ax_main_g = axes[0, 0], axes[0, 1]
ax_res_s, ax_res_g = axes[1, 0], axes[1, 1]

# =============================================================================
# Sigma variation (left column)
# =============================================================================
colors_sigma = np.array(plt.colormaps["Reds"](np.linspace(0, 1, len(sigma)+2)))[1:-1]

for i in range(len(sigma)):
    ax_main_s.plot(xx, V_sigma[i], color=colors_sigma[i], linewidth=2.0, linestyle='-')

for i in range(len(sigma)):
    residual = V_sigma[i] / V_sigma_ref - 1
    ax_res_s.plot(xx, residual, color=colors_sigma[i], linewidth=2.0, linestyle='-')

# Legend
param_handles_s = []
for i, s_val in enumerate(sigma):
    param_handles_s.append(Line2D([0], [0], color=colors_sigma[i], lw=6, label=rf"$\sigma={s_val:.3g}$"))
leg2_s = ax_main_s.legend(handles=param_handles_s, loc="upper left", fontsize=16)
ax_main_s.add_artist(leg2_s)

# Styling
ax_main_s.set_title(r"Voigt profile variation with $\sigma$", pad=10)
ax_main_s.set_ylabel(r"$V(x;\ \sigma,\ \gamma_0)$")
ax_main_s.set_yscale("log")
V_min_s, V_max_s = V_sigma.min(), V_sigma.max()
ax_main_s.set_ylim(V_min_s * 0.5, V_max_s * 2.0)

ax_res_s.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res_s.set_xlabel("x")
ax_res_s.set_ylabel("residual")
ax_res_s.set_yscale("symlog", linthresh=1e-6)
major_ticks_s = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res_s.set_yticks(major_ticks_s)
ax_res_s.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res_s.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

# =============================================================================
# Gamma variation (right column)
# =============================================================================
colors_gamma = np.array(plt.colormaps["Greens"](np.linspace(0, 1, len(gamma)+2)))[1:-1]

for i in range(len(gamma)):
    ax_main_g.plot(xx, V_gamma[i], color=colors_gamma[i], linewidth=2.0, linestyle='-')

for i in range(len(gamma)):
    residual = V_gamma[i] / V_gamma_ref - 1
    ax_res_g.plot(xx, residual, color=colors_gamma[i], linewidth=2.0, linestyle='-')

# Legend
param_handles_g = []
for i, g_val in enumerate(gamma):
    param_handles_g.append(Line2D([0], [0], color=colors_gamma[i], lw=6, label=rf"$\gamma={g_val:.4g}$"))
leg2_g = ax_main_g.legend(handles=param_handles_g, loc="upper left", fontsize=16)
ax_main_g.add_artist(leg2_g)

# Styling
ax_main_g.set_title(r"Voigt profile variation with $\gamma$", pad=10)
ax_main_g.set_ylabel(r"$V(x;\ \sigma_0,\ \gamma)$")
ax_main_g.set_yscale("log")
V_min_g, V_max_g = V_gamma.min(), V_gamma.max()
ax_main_g.set_ylim(V_min_g * 0.5, V_max_g * 2.0)

ax_res_g.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res_g.set_xlabel("x")
ax_res_g.set_ylabel("residual")
ax_res_g.set_yscale("symlog", linthresh=1e-6)
major_ticks_g = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res_g.set_yticks(major_ticks_g)
ax_res_g.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res_g.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

# =============================================================================
fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.3)  # Add wspace for column separation

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

ax = axes[0]
ax.plot(xx, dV_dx)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dx")
ax.set_title("Derivative wrt x")

ax = axes[1]
ax.plot(xx, dV_dsigma)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dσ")
ax.set_title("Derivative wrt σ")

ax = axes[2]
ax.plot(xx, dV_dgamma)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dγ")
ax.set_title("Derivative wrt γ")

fig.tight_layout()
plt.show()

# Lines

In [ ]:
from skewerjax.constants.lines import LINES, get_line, LineBundle

### Basic Usage: Fetch a single line by ID

In [ ]:
# Get the Lyman-alpha transition
lya = get_line("HI-LyA")

print(f"ID: {lya.id}")
print(f"Species: {lya.species}")
print(f"Name: {lya.name}")
print(f"Rest wavelength: {lya.lambda0_A:.4f} Å")
print(f"Oscillator strength: {lya.f_osc:.4f}")
print(f"Einstein coefficient: {lya.A21_Hz:.4e} Hz = {lya.A21_GHz:.4f} GHz")
print(f"Mass: {lya.mass_amu:.4f} amu")

### Explore the Global LINES Database

In [ ]:
print(f"Total lines loaded: {len(LINES)}")

for line_id, line in sorted(LINES.items(), key=lambda x: x[1].lambda0_A):
    print(f"  Name:{line_id:11s} λ={line.lambda0_A:8.4f} Å. Species: {line.species:7s} f={line.f_osc:.4f} ")

### Load some selected absorption lines using the LineBundle class

In [ ]:
line_bundle = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

print(f"Lines: {line_bundle.n_lines}, Species: {line_bundle.n_species}")
print(f"Species keys: {line_bundle.species_keys}")

for ii in range(line_bundle.n_lines):
    print(f"{ii:2d} {line_bundle.lines[ii].id} {line_bundle.lines[ii].species} {line_bundle.lines[ii].lambda0_A} {line_bundle.lines[ii].f_osc} {line_bundle.lines[ii].A21_GHz} {line_bundle.lines[ii].mass_amu}")

print(f"HI lines: {line_bundle.lines_for_species('HI')}")
print(f"SiIII lines: {line_bundle.lines_for_species('SiIII')}")

print(f"HI indices: {line_bundle.indices_for_species('HI')}")
print(f"SiIII indices: {line_bundle.indices_for_species('SiIII')}")

print(f"Lambda0: {line_bundle.lambda0_A}")
print(f"A21: {line_bundle.A21_GHz}")
print(f"Mass: {line_bundle.mass_amu}")

# Fields

In [ ]:
%autoreload

In [ ]:
from skewerjax.fields.fields import ScalarField, VectorField, Fields

In [ ]:
def temperature_sampler(T0_min: float = 1e0, T0_max: float = 1e6, L_box: float = 1.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        # Capture batch dimensions (e.g., (7,5) from (7,5,4))
        *batch_dims, _ = x.shape
        # Flatten all but last dimension: (..., 4) -> (N_points, 4)
        x_flat = x.reshape(-1, 4)
        # Compute on flat array
        z = x_flat[:, -1]  # z-position
        log_T0_min = jnp.log10(T0_min)
        log_T0_max = jnp.log10(T0_max)
        T_flat = 10**(log_T0_min + (log_T0_max - log_T0_min) * (z / L_box))
        # Restore original batch shape
        return T_flat.reshape(batch_dims)
    return _sample


def velocity_sampler(v0_min: float = -0.001, v0_max: float = 0.001, L_box: float = 1.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        *batch_dims, _ = x.shape
        x_flat = x.reshape(-1, 4)
        # Extract x-position and compute vx
        x_pos = x_flat[:, 1]
        vx_flat = v0_min + (v0_max - v0_min) * x_pos / L_box
        # Build 3D vector: (N_points, 3)
        v_flat = jnp.stack([vx_flat, jnp.zeros_like(vx_flat), jnp.zeros_like(vx_flat)], axis=-1)
        # Restore batch shape + vector dimension: (..., 3)
        return v_flat.reshape(*batch_dims, 3)
    return _sample


def turbulence_sampler(b_turb: float = 0.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        *batch_dims, _ = x.shape
        x_flat = x.reshape(-1, 4)
        result_flat = jnp.full(x_flat.shape[0], b_turb)
        return result_flat.reshape(batch_dims)
    return _sample


def collisional_sampler(A_coll: float = 0.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        *batch_dims, _ = x.shape
        x_flat = x.reshape(-1, 4)
        result_flat = jnp.full(x_flat.shape[0], A_coll)
        return result_flat.reshape(batch_dims)
    return _sample


def number_density_sampler(n0_min: float = 1e0, n0_max: float = 1e6, L_box: float = 1.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        *batch_dims, _ = x.shape
        x_flat = x.reshape(-1, 4)
        y = x_flat[:, 2]  # y-position
        log_n0_min = jnp.log10(n0_min)
        log_n0_max = jnp.log10(n0_max)
        n_flat = 10**(log_n0_min + (log_n0_max - log_n0_min) * (y / L_box))
        return n_flat.reshape(batch_dims)
    return _sample


L_box = 1.0

fields = Fields({
    "HI": {
        "temperature": ScalarField(temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)),
        "velocity": VectorField(velocity_sampler(v0_min=-0.001, v0_max=0.001, L_box=L_box)),
        "turbulence": ScalarField(turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(number_density_sampler(n0_min=1e0, n0_max=1e6, L_box=L_box)),
    },
    "SiIII": {
        "temperature": ScalarField(temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)),
        "velocity": VectorField(velocity_sampler(v0_min=-0.001, v0_max=0.001, L_box=L_box)),
        "turbulence": ScalarField(turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(number_density_sampler(n0_min=1e0, n0_max=1e6, L_box=L_box)),
    }
})

In [ ]:
def make_y_plane_points(L_box, y_plane, scale_factor, n_z, n_x):
    """
    Plane y = y_plane, varying (z, x).

    Returns
    -------
    points : (n_z, n_x, 4)   # [a, x, y, z]
    x_grid : (n_z, n_x)
    z_grid : (n_z, n_x)
    """
    z = jnp.linspace(0.0, L_box, n_z)
    x = jnp.linspace(0.0, L_box, n_x)
    z_grid, x_grid = jnp.meshgrid(z, x, indexing="ij")

    y_grid = jnp.full_like(z_grid, y_plane)
    a_grid = jnp.full_like(z_grid, scale_factor)

    points = jnp.stack([a_grid, x_grid, y_grid, z_grid], axis=-1)
    return points, x_grid, z_grid


def make_x_plane_points(L_box, x_plane, scale_factor, n_z, n_y):
    """
    Plane x = x_plane, varying (z, y).

    Returns
    -------
    points : (n_z, n_y, 4)   # [a, x, y, z]
    y_grid : (n_z, n_y)
    z_grid : (n_z, n_y)
    """
    z = jnp.linspace(0.0, L_box, n_z)
    y = jnp.linspace(0.0, L_box, n_y)
    z_grid, y_grid = jnp.meshgrid(z, y, indexing="ij")

    x_grid = jnp.full_like(z_grid, x_plane)
    a_grid = jnp.full_like(z_grid, scale_factor)

    points = jnp.stack([a_grid, x_grid, y_grid, z_grid], axis=-1)
    return points, y_grid, z_grid


def make_LoS(batch_shape, direction):
    """
    Make normalized line-of-sight directions with given base direction.
    batch_shape: e.g. (N_sk, N_ppsk)
    direction  : iterable of length 3
    """
    base = jnp.asarray(direction, dtype=float)
    LoS = jnp.broadcast_to(base, tuple(batch_shape) + (3,))
    return LoS / jnp.linalg.norm(LoS, axis=-1, keepdims=True)

In [ ]:
print("Available species:", fields.species_keys)
print("Available fields:", fields.field_keys)

scale_factor = 1.0
y_plane = L_box / 2.0
key_species = "HI"
i_species = fields.species_keys.index(key_species)

# ================================================================
# Dense grid for imshow (temperature background)
# ================================================================
N_grid = 64
points_imshow, x_full_imshow, z_full_imshow = make_y_plane_points(
    L_box, y_plane, scale_factor, n_z=N_grid, n_x=N_grid
)
T_imshow = fields.temperature(points_imshow)  # (N_species, N_grid, N_grid)
print("\nT_imshow shape:", T_imshow.shape, "(N_species, N_grid, N_grid)")

# ================================================================
# Sparse skewers + LoS sampling
# ================================================================
N_sk, N_ppsk = 7, 5
points, x_full, z_full = make_y_plane_points(
    L_box, y_plane, scale_factor, n_z=N_sk, n_x=N_ppsk
)
LoS_directions = make_LoS(points.shape[:-1], [1.0, 0.0, 0.0])

sampled_fields = fields.sample(points, LoS_directions)
print("\n=== Shape validation ===")
for species_name, species_data in sampled_fields.items():
    for field_name, values in species_data.items():
        print(f"{species_name}.{field_name}: {values.shape}")

# ================================================================
# Plot
# ================================================================
fig, ax = plt.subplots(figsize=(10, 8))

# Background: log10 T
logT = np.log10(np.array(T_imshow[i_species]))
x_min, x_max = float(x_full_imshow.min()), float(x_full_imshow.max())
z_min, z_max = float(z_full_imshow.min()), float(z_full_imshow.max())

im = ax.imshow(
    logT,
    origin="lower",
    extent=(x_min, x_max, z_min, z_max),
    aspect="auto",
    cmap="inferno",
)
cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.046)
cbar.set_label(r"$\log_{10} T\ \mathrm{[K]}$")

# Sampling points and connecting lines
x_full_np = np.array(x_full)
z_full_np = np.array(z_full)
LoS_np = np.array(LoS_directions)

ax.scatter(
    x_full_np.T, z_full_np.T,
    c="tab:red", marker="o", s=35, alpha=0.9,
    edgecolor="k", linewidth=0.5,
)
ax.plot(
    x_full_np.T, z_full_np.T,
    color="tab:red", linewidth=0.8, alpha=0.9,
)

# LoS directions (red)
ax.quiver(
    x_full_np.T, z_full_np.T,
    LoS_np[:, :, 0].T, LoS_np[:, :, 2].T,
    color="tab:red", angles="xy", scale_units="xy",
    scale=None, width=0.003, alpha=0.9,
)

# Gas velocities (blue, xz components)
vx = np.array(sampled_fields[key_species]["velocity"][:, :, 0])
vz = np.array(sampled_fields[key_species]["velocity"][:, :, 2])

ax.quiver(
    x_full_np.T, z_full_np.T,
    vx.T, vz.T,
    color="tab:blue", angles="xy", scale_units="xy",
    scale=None, width=0.007, alpha=0.9,
)

# Domain frame & limits
ax.plot(
    [x_min, x_max, x_max, x_min, x_min],
    [z_min, z_min, z_max, z_max, z_min],
    color="k", linewidth=2.5, alpha=0.9,
)
limit_padding = 0.2
ax.set_xlim(x_min - limit_padding, x_max + limit_padding)
ax.set_ylim(z_min - limit_padding, z_max + limit_padding)

# Labels, title, grid
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$z$")
ax.set_title(rf"{key_species} slice at $y={float(y_plane):.3g}$", pad=12)
ax.grid(color="w", linewidth=0.8, alpha=0.15)

# Label each skewer with its (log10) temperature
T_sample = np.array(sampled_fields[key_species]["temperature"])  # (N_sk, N_ppsk)
dz = (z_max - z_min)

for i in range(N_sk):
    z_i = float(z_full_np[i, 0])
    x_center = float(x_full_np[i, :].mean())
    logT_i = np.log10(T_sample[i, 0])  # assume constant along skewer

    ax.text(
        x_center, z_i + 0.02 * dz,
        rf"$\log_{{10}} T = {logT_i:.2f}$",
        color="k", ha="center", va="bottom", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="k", alpha=0.0),
    )

# Legend
legend_handles = [
    Line2D([0], [0], marker="o", linestyle="none",
           markerfacecolor="tab:red", markeredgecolor="k",
           markersize=8, label="sampling points"),
    Line2D([0], [0], color="tab:red", lw=2, label="LoS direction"),
    Line2D([0], [0], color="tab:blue", lw=2, label="gas velocity"),
]
leg = ax.legend(
    handles=legend_handles,
    loc="lower center",
    frameon=True,
    fontsize=14,
    framealpha=0.9,
    ncols=3,
)
leg.get_frame().set_edgecolor("k")

plt.tight_layout()
plt.show()


In [ ]:
x_plane = L_box / 2.0
key_species = "HI"
i_species = fields.species_keys.index(key_species)

# ================================================================
# Dense grid for imshow (number density background) in x-plane
# ================================================================
N_grid = 64
points_imshow_x, y_full_imshow_x, z_full_imshow_x = make_x_plane_points(
    L_box, x_plane, scale_factor, n_z=N_grid, n_y=N_grid
)
n_imshow = fields.number_density(points_imshow_x)  # (N_species, N_grid, N_grid)

# ================================================================
# Sparse skewers + LoS sampling in x-plane
# ================================================================
N_sk, N_ppsk = 7, 5
points_x, y_full_x, z_full_x = make_x_plane_points(
    L_box, x_plane, scale_factor, n_z=N_sk, n_y=N_ppsk
)
LoS_directions_x = make_LoS(points_x.shape[:-1], [0.0, 1.0, 0.0])  # along +y

sampled_fields_x = fields.sample(points_x, LoS_directions_x)

# ================================================================
# Plot number density slice
# ================================================================
fig2, ax2 = plt.subplots(figsize=(10, 8))

# Background: log10 n
logn = np.log10(np.array(n_imshow[i_species]))
y_min, y_max = float(y_full_imshow_x.min()), float(y_full_imshow_x.max())
z_min, z_max = float(z_full_imshow_x.min()), float(z_full_imshow_x.max())

im2 = ax2.imshow(
    logn,
    origin="lower",
    extent=(y_min, y_max, z_min, z_max),
    aspect="auto",
    cmap="inferno",
)
cbar2 = fig2.colorbar(im2, ax=ax2, pad=0.02, fraction=0.046)
cbar2.set_label(r"$\log_{10} n\ \mathrm{[units]}$")

# Sampling points and connecting lines (skewers in y)
y_full_x_np = np.array(y_full_x)
z_full_x_np = np.array(z_full_x)
LoS_x_np    = np.array(LoS_directions_x)

ax2.scatter(
    y_full_x_np.T, z_full_x_np.T,
    c="tab:red", marker="o", s=100, alpha=0.9,
    edgecolor="k", linewidth=0.5,
)
ax2.plot(
    y_full_x_np.T, z_full_x_np.T,
    color="tab:red", linewidth=0.8, alpha=0.9,
)

# LoS directions in (y,z) plane
ax2.quiver(
    y_full_x_np.T, z_full_x_np.T,
    LoS_x_np[:, :, 1].T, LoS_x_np[:, :, 2].T,   # y, z components
    color="tab:red", angles="xy", scale_units="xy",
    scale=None, width=0.003, alpha=0.9,
)

# Gas velocities (yz components)
vy = np.array(sampled_fields_x[key_species]["velocity"][:, :, 1])
vz = np.array(sampled_fields_x[key_species]["velocity"][:, :, 2])

ax2.quiver(
    y_full_x_np.T, z_full_x_np.T,
    vy.T, vz.T,
    color="tab:blue", angles="xy", scale_units="xy",
    scale=None, width=0.007, alpha=0.9,
)

# Domain frame & limits
ax2.plot(
    [y_min, y_max, y_max, y_min, y_min],
    [z_min, z_min, z_max, z_max, z_min],
    color="k", linewidth=2.5, alpha=0.9,
)
limit_padding = 0.2
ax2.set_xlim(y_min - limit_padding, y_max + limit_padding)
ax2.set_ylim(z_min - limit_padding, z_max + limit_padding)

# Labels, title, grid
ax2.set_xlabel(r"$y$")
ax2.set_ylabel(r"$z$")
ax2.set_title(
    rf"{key_species} number density slice at $x={float(x_plane):.3g}$",
    pad=12,
)
ax2.grid(color="w", linewidth=0.8, alpha=0.15)

# Legend
legend_handles2 = [
    Line2D([0], [0], marker="o", linestyle="none",
           markerfacecolor="tab:red", markeredgecolor="k",
           markersize=8, label="sampling points"),
    Line2D([0], [0], color="tab:red", lw=2, label="LoS direction"),
    Line2D([0], [0], color="tab:blue", lw=2, label="gas velocity"),
]
leg2 = ax2.legend(
    handles=legend_handles2,
    loc="lower center",
    frameon=True,
    fontsize=14,
    framealpha=0.9,
    ncols=3,
)
leg2.get_frame().set_edgecolor("k")

plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------
# Setup & sampling
# ---------------------------------------------------------
key_species = "HI"
i_species = fields.species_keys.index(key_species)

scale_factor = 1.0
L = float(L_box)

# Grid resolution for skewers
N_sk   = 7   # number of skewers (along z)
N_ppsk = 5   # points per skewer (along x or y)

# Unified colors
los_line_color   = "k"          # skewer lines + LoS arrows
vel_arrow_color  = "tab:blue"   # all velocity arrows

# ----------------------
# Y-plane (temperature)
# ----------------------
y_plane = L_box / 2.0
points_y, x_y_jax, z_y_jax = make_y_plane_points(
    L_box, y_plane, scale_factor, n_z=N_sk, n_x=N_ppsk
)
LoS_y_jax = make_LoS(points_y.shape[:-1], [1.0, 0.0, 0.0])  # LoS along +x

# Sample T and velocity for y-plane
T_y   = np.array(fields.temperature(points_y)[i_species])    # (N_sk, N_ppsk)
vel_y = np.array(fields.velocity(points_y)[i_species])       # (N_sk, N_ppsk, 3)

# Convert coords & LoS to numpy
x_y = np.array(x_y_jax)
z_y = np.array(z_y_jax)
y_y = np.full_like(x_y, float(y_plane))
LoS_y = np.array(LoS_y_jax)

# Colors for y-plane points: log10(T)
logT_y = np.log10(T_y)
norm_T = mpl.colors.Normalize(vmin=logT_y.min(), vmax=logT_y.max())
cmap_T = mpl.cm.Reds

# ----------------------
# X-planes (number density) at 0.1L and 0.9L
# ----------------------
x_planes = [0.5 * L_box]
x_plane_data = []  # list of dicts with plane data

for xp in x_planes:
    points_x, y_x_jax, z_x_jax = make_x_plane_points(
        L_box, xp, scale_factor, n_z=N_sk, n_y=N_ppsk
    )
    LoS_x_jax = make_LoS(points_x.shape[:-1], [0.0, 1.0, 0.0])  # LoS along +y

    n_x   = np.array(fields.number_density(points_x)[i_species])  # (N_sk, N_ppsk)
    vel_x = np.array(fields.velocity(points_x)[i_species])        # (N_sk, N_ppsk, 3)

    x_x = np.full_like(np.array(y_x_jax), float(xp))
    y_x = np.array(y_x_jax)
    z_x = np.array(z_x_jax)
    LoS_x = np.array(LoS_x_jax)

    x_plane_data.append(
        dict(xp=float(xp), x=x_x, y=y_x, z=z_x, LoS=LoS_x, n=n_x, vel=vel_x)
    )

# Colors for x-plane points: log10(n), normalized over *both* planes
logn_all = np.concatenate(
    [np.log10(d["n"]).ravel() for d in x_plane_data]
)
norm_n = mpl.colors.Normalize(vmin=logn_all.min(), vmax=logn_all.max())
cmap_n = mpl.cm.Greens

# ---------------------------------------------------------
# 3D plot
# ---------------------------------------------------------
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

# ----------------------
# Y-plane: points + lines + arrows
# ----------------------
# Flatten for scatter
x_y_f     = x_y.ravel()
y_y_f     = y_y.ravel()
z_y_f     = z_y.ravel()
logT_y_f  = logT_y.ravel()
colors_y  = cmap_T(norm_T(logT_y_f))

ax.scatter(
    x_y_f, y_y_f, z_y_f,
    c=colors_y, marker="o", s=100, alpha=0.9, edgecolor="k", linewidth=0.5,
)

# Join points along each skewer (fixed z index)
for i in range(N_sk):
    ax.plot(
        x_y[i, :], y_y[i, :], z_y[i, :],
        color=los_line_color, linewidth=1.0, alpha=0.9
    )

# Subsample for arrows
N_sk_y, N_ppsk_y, _ = vel_y.shape
step_z_y = max(1, N_sk_y  // 5)
step_x_y = max(1, N_ppsk_y // 5)

Xq_y = x_y[::step_z_y, ::step_x_y]
Yq_y = y_y[::step_z_y, ::step_x_y]
Zq_y = z_y[::step_z_y, ::step_x_y]

U_LoS_y = LoS_y[::step_z_y, ::step_x_y, 0]
V_LoS_y = LoS_y[::step_z_y, ::step_x_y, 1]
W_LoS_y = LoS_y[::step_z_y, ::step_x_y, 2]

U_v_y = vel_y[::step_z_y, ::step_x_y, 0]
V_v_y = vel_y[::step_z_y, ::step_x_y, 1]
W_v_y = vel_y[::step_z_y, ::step_x_y, 2]

# LoS arrows (y-plane) – all black
ax.quiver(
    Xq_y, Yq_y, Zq_y,
    U_LoS_y, V_LoS_y, W_LoS_y,
    length=0.07 * L, normalize=True, color=los_line_color, linewidth=1.0, alpha=0.9
)

# 3D velocity arrows (y-plane) – all same color
ax.quiver(
    Xq_y, Yq_y, Zq_y,
    U_v_y, V_v_y, W_v_y,
    length=0.07 * L, normalize=True, color=vel_arrow_color, linewidth=1.5, alpha=0.9,
)

# ----------------------
# X-planes: points + lines + arrows
# ----------------------
# markers for first/second x-plane
x_plane_markers = ["^", "s"]

for d, mkr in zip(x_plane_data, x_plane_markers):
    x_x   = d["x"]
    y_x   = d["y"]
    z_x   = d["z"]
    LoS_x = d["LoS"]
    n_x   = d["n"]
    vel_x = d["vel"]

    # Colors from log10(n) with shared norm
    logn_x   = np.log10(n_x)
    colors_x = cmap_n(norm_n(logn_x.ravel()))

    ax.scatter(
        x_x.ravel(), y_x.ravel(), z_x.ravel(),
        c=colors_x, marker=mkr, s=100, alpha=0.9, edgecolor="k", linewidth=0.5,
    )

    # Join points along skewers (fixed z index)
    for i in range(N_sk):
        ax.plot(
            x_x[i, :], y_x[i, :], z_x[i, :],
            color=los_line_color, linewidth=1.0, alpha=0.9
        )

    # Subsample for arrows
    N_sk_x, N_ppsk_x, _ = vel_x.shape
    step_z_x = max(1, N_sk_x  // 5)
    step_y_x = max(1, N_ppsk_x // 5)

    Xq_x = x_x[::step_z_x, ::step_y_x]
    Yq_x = y_x[::step_z_x, ::step_y_x]
    Zq_x = z_x[::step_z_x, ::step_y_x]

    U_LoS_x = LoS_x[::step_z_x, ::step_y_x, 0]
    V_LoS_x = LoS_x[::step_z_x, ::step_y_x, 1]
    W_LoS_x = LoS_x[::step_z_x, ::step_y_x, 2]

    U_v_x = vel_x[::step_z_x, ::step_y_x, 0]
    V_v_x = vel_x[::step_z_x, ::step_y_x, 1]
    W_v_x = vel_x[::step_z_x, ::step_y_x, 2]

    # LoS arrows (x-plane) – all black
    ax.quiver(
        Xq_x, Yq_x, Zq_x, U_LoS_x, V_LoS_x, W_LoS_x,
        length=0.07 * L, normalize=True, color=los_line_color, linewidth=1.0, alpha=0.9
    )

    # 3D velocity arrows (x-plane) – same color as y-plane
    ax.quiver(
        Xq_x, Yq_x, Zq_x, U_v_x, V_v_x, W_v_x,
        length=0.07 * L, normalize=True, color=vel_arrow_color, linewidth=1.5, alpha=0.9
    )

# ----------------------
# Intersection lines for orientation (y-plane ∩ each x-plane)
# ----------------------
z_line = np.linspace(0.0, L, 100)
for xp in x_planes:
    x_line = np.full_like(z_line, float(xp))
    y_line = np.full_like(z_line, float(y_plane))
    ax.plot(x_line, y_line, z_line, color="k", linewidth=1.5, alpha=0.5)

# ----------------------
# Colorbars (place them in separate axes with padding)
# ----------------------
sm_T = mpl.cm.ScalarMappable(norm=norm_T, cmap=cmap_T)
sm_T.set_array([])

sm_n = mpl.cm.ScalarMappable(norm=norm_n, cmap=cmap_n)
sm_n.set_array([])

# Manually create two vertical colorbar axes on the right
# [left, bottom, width, height] in figure coordinates
cax_T = fig.add_axes([0.88, 0.15, 0.02, 0.30])  # lower bar
cax_n = fig.add_axes([0.88, 0.55, 0.02, 0.30])  # upper bar

cbar_T = fig.colorbar(sm_T, cax=cax_T)
cbar_T.set_label(r"$\log_{10} T\ \mathrm{[K]}$")

cbar_n = fig.colorbar(sm_n, cax=cax_n)
cbar_n.set_label(r"$\log_{10} n\ \mathrm{[nm^{-3}]}$")

# ----------------------
# Axes, aspect, limits, cube, ticks, planes, legend
# ----------------------
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_zlabel(r"$z$")
ax.set_title(rf"{key_species} 3D skewers", pad=12)

ax.grid(False)

pad = 0.05 * L
ax.set_xlim(-pad, L + pad)
ax.set_ylim(-pad, L + pad)
ax.set_zlim(-pad, L + pad)

# fewer ticks per axis
ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
ax.zaxis.set_major_locator(mpl.ticker.MaxNLocator(4))

# remove axis planes (background panes)
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    try:
        axis.pane.set_visible(False)
    except AttributeError:
        # for older Matplotlib versions without 'pane'
        pass

# Equal-ish aspect
ax.set_box_aspect((L + 2*pad, L + 2*pad, L + 2*pad))

# Camera view
ax.view_init(elev=28, azim=-55)

# ----------------------
# Legend: only scatter symbols for the planes
# ----------------------
legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="none",
        markerfacecolor="0.7", markeredgecolor="k",
        markersize=8, label=rf"y-plane, $y={float(y_plane):.3g}$ (T)",
    ),
    Line2D(
        [0], [0], marker="^", linestyle="none",
        markerfacecolor="0.7", markeredgecolor="k",
        markersize=8, label=rf"x-plane, $x={0.5*L_box:.3g}$ (n)",
    ),
]

leg = ax.legend(
    handles=legend_handles,
    loc="upper left",
    fontsize=16,
    frameon=True,
    framealpha=0.9,
)
leg.get_frame().set_edgecolor("k")

plt.tight_layout()
plt.show()

# Line Profile

In [ ]:
from skewerjax.optical_depth.optical_depth import compute_line_profiles

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PhysicalConstants:
    c: float = 2.99792458e8                         # speed of light      — UNITS: [m / s]
    kB_amu: float = 8314.46262102654                # Boltzman constant   — UNITS: [amu ⋅ m^2 ⋅ s^-2 ⋅ K^-1]
    qe: float = 1.60217663e-19                      # electron charge     — UNITS: [C]
    me: float = 9.1093837e-31                       # electron mass       — UNITS: [kg]
    epsilon_0: float = 8.85418782e-12               # vacuum permittivity — UNITS: [C^2 ⋅ kg^−1 ⋅ m^−3 ⋅ s^2]

# single shared instance (immutable)
phys = PhysicalConstants()

In [ ]:
print(line_bundle.n_lines)
print(line_bundle.n_species)

for key_species in line_bundle.species_keys:
    print(key_species)
    print(line_bundle.indices_for_species(key_species))

In [ ]:
lambda0_A = line_bundle.lambda0_A       # (N_lines,) — UNITS: [Angs]
A21_GHz   = line_bundle.A21_GHz         # (N_lines,) — UNITS: [GHz]
mass_amu  = line_bundle.mass_amu        # (N_lines,) — UNITS: [amu]
f_osc     = line_bundle.f_osc           # (N_lines,) — UNITS: [adim]

print(lambda0_A.shape)

In [ ]:
collisional   = fields.collisional(points)                   # (N_species, *batch_shape)
temperature   = fields.temperature(points)                   # (N_species, *batch_shape)
turbulence    = fields.turbulence(points)                    # (N_species, *batch_shape)
LoS_velocity  = fields.LoS_velocity(points, LoS_directions)  # (N_species, *batch_shape)

print(collisional.shape)

In [ ]:
gamma_GHz = (A21_GHz[:, None, None] + A_coll_GHz) / (4.0 * jnp.pi)         # (N_sp, 1, 1) — UNITS: [GHz]

In [ ]:
N_sp = 3     # number of absroption lines ''species'' (considering different absorption lines from different species at the same time)
N_sk = 7     # number of skerwers
N_ppsk = 5   # number of points sampled per skerwer

scale_factor = jnp.ones((N_sk, N_ppsk))                                                          #       (N_sk, N_ppsk)            — UNITS: adim

v_over_c_LoS = jnp.ones((N_sp, N_sk, N_ppsk))*jnp.linspace(-0.001, 0.001, N_ppsk)[None, None, :] # (N_sp, N_sk, N_ppsk)            — UNITS: adim
T_K = jnp.ones((N_sp, N_sk, N_ppsk))*jnp.geomspace(1e0, 1e6, N_sk)[None, :, None]                # (N_sp, N_sk, N_ppsk)            — UNITS: K
A_coll_GHz = jnp.zeros((N_sp, N_sk, N_ppsk))                                                     # (N_sp, N_sk, N_ppsk)            — UNITS: GHz
b_turb_m_per_s = jnp.zeros((N_sp, N_sk, N_ppsk))                                                 # (N_sp, N_sk, N_ppsk)            — UNITS: m/s

lambdas_A = jnp.linspace(970, 1280, 2**18)                                                       #                     (N_lambda,) — UNITS: Å

number_density_inv_nm3 = jnp.ones((N_sp, N_sk, N_ppsk))                                          # (N_sp, N_sk, N_ppsk)            — UNITS: 1/nm³

In [ ]:
lambda0_A = line_bundle.lambda0_A       # (N_sp,) — UNITS: [Angs]
A21_GHz   = line_bundle.A21_GHz         # (N_sp,) — UNITS: [GHz]
mass_amu  = line_bundle.mass_amu        # (N_sp,) — UNITS: [amu]
f_osc     = line_bundle.f_osc           # (N_sp,) — UNITS: [adim]

gamma_GHz = (A21_GHz[:, None, None] + A_coll_GHz) / (4.0 * jnp.pi)         # (N_sp, 1, 1) — UNITS: [GHz]

v_th_m_per_s = jnp.sqrt(2.0 * phys.kB_amu * T_K / mass_amu[:, None, None]) # (N_sp, N_sk, N_ppsk) — UNITS: [m/s]
v_eff_A_GHz = (v_th_m_per_s + b_turb_m_per_s)*10                           # (N_sp, N_sk, N_ppsk) — UNITS: [Å/ns]
sigma_GHz = v_eff_A_GHz / (2.0 * lambda0_A[:, None, None])                 # (N_sp, N_sk, N_ppsk) — UNITS: [GHz]

lambdas_A_prime = lambdas_A[None, None, None, :] * scale_factor[None, ..., None] * (1 + v_over_c_LoS[..., None])
nu_GHz = ( (1. / lambdas_A_prime) - (1. / lambda0_A[:, None, None, None]) ) * phys.c*10  # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [GHz]

phi_ns = compute_line_profiles(                            # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [ns]
    nu_GHz, sigma_GHz, gamma_GHz,
    mode="ExoJAX", compute_mode="pmap", num_gpus=None
)

# print quantities to check their numeric range (avoid potential numerical overflows)
print("nu", nu_GHz.shape, nu_GHz.min(), nu_GHz.max())
print("gamma", gamma_GHz.shape, gamma_GHz.min(), gamma_GHz.max())
print("sigma", sigma_GHz.shape, sigma_GHz.min(), sigma_GHz.max())
print("phi", phi_ns.shape, phi_ns.min(), phi_ns.max())

In [ ]:
cte_nm2_per_ns = (phys.qe**2 / (4 * phys.epsilon_0 * phys.me * phys.c))*1e3                   # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [nm² / ns]
sigma_nm2 = cte_nm2_per_ns * f_osc[:, None, None, None] * phi_ns                              # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [nm²]
integrand = sigma_nm2 * number_density_inv_nm3[..., None] * scale_factor[None, ..., None]**-2 # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [1/nm]

print("integrand", integrand.shape, integrand.min(), integrand.max())

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(lambdas_A)           # (N_lambda,)
phi_np = np.asarray(phi_ns)                     # (N_sp, N_sk, N_ppsk, N_lambda)
T_vals = np.asarray(T_K[0, :, 0])            # (N_sk,)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_ppsk = N_ppsk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ax.scatter(np.array([line.lambda0_A]), np.array([4./line.A21_GHz]), color="grey", s=50, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_sk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for temperatures (colors)
temp_handles = []
for i_sk in range(N_sk):
    lbl = rf"$T = {T_vals[i_sk]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)

# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=N_sp)

fig.tight_layout()
plt.show()



# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=N_sp, figsize=(14, 6), sharex=False, sharey=True)
if N_sp == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    # Plot each temperature (each skewer)
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_sk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(lambdas_A)                          # (N_lambda,)
phi_np = np.asarray(phi_ns)                                    # (N_sp, N_sk, N_ppsk, N_lambda)
v_over_c_LoS_np = np.asarray(v_over_c_LoS[0, 0])            # (N_ppsk,)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_sk = N_sk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ax.scatter(np.array([line.lambda0_A]), np.array([4./line.A21_GHz]), color="grey", s=50, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_ppsk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for velocities (colors)
temp_handles = []
for i_ppsk in range(N_ppsk):
    lbl = rf"$v_\mathrm{{LOS}}/c = {v_over_c_LoS_np[i_ppsk]:.2g}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_ppsk], lw=4, linestyle="-", label=lbl))
leg_v = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_v)

# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=N_sp)

fig.tight_layout()
plt.show()


# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=N_sp, figsize=(14, 6), sharex=False, sharey=True)
if N_sp == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    # Plot each temperature (each skewer)
    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_ppsk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()


# tests to explore vmap, pmap and Pytrees

In [ ]:
from jax import vmap, jit, pmap
import jax.numpy as jnp
import matplotlib.pyplot as plt



def make_chi2_computer(model_fn):
    """Factory: model_fn is baked into the returned function."""
    def chi2(data, params):
        model = model_fn(data["x"], params)
        return jnp.sum((data["y"] - model)**2 / data["noise"]**2)
    return chi2

def sinusoid_model(x, params):
    return params["amp"] * jnp.sin(x + params["phase"])


chi2_sinusoid = make_chi2_computer(sinusoid_model)



# Each sightline is a dict of arrays shape (n_points,)
datasets = {
    "x": jnp.array([
        [0.0, 0.5, 1.0, 1.5, 2.0],  # Sightline 0
        [0.0, 0.7, 1.4, 2.1, 2.8],  # Sightline 1
        [0.0, 1.0, 2.0, 3.0, 4.0],  # Sightline 2
    ]),
    "y": jnp.array([
        [0.1, 0.9, 0.2, -0.7, -0.3],  # Observed flux for sightline 0
        [0.0, 0.5, 0.8, 0.3, -0.5],   # Sightline 1
        [0.1, 1.2, 0.9, 0.4, 0.1],    # Sightline 2
    ]),
    "noise": jnp.full((3, 5), 0.05)  # Constant noise for all
}

params_sine = {"amp": 1.0, "phase": 0.0}




vcompute_sinusoid = vmap(chi2_sinusoid, in_axes=(0, None))
chi2_vmap = vcompute_sinusoid(datasets, params_sine)
print(chi2_vmap)

In [ ]:
import jax

In [ ]:
# Simulate 8 GPUs (use real number if available)
n_devices = min(8, jax.local_device_count())
print(f"Available devices: {jax.local_device_count()}, Using: {n_devices}")

# Reshape data: (n_devices, n_sightlines_per_device, ...)
# Pad to multiple of n_devices if needed
n_total = datasets["x"].shape[0]
if n_total % n_devices != 0:
    pad_size = n_devices - (n_total % n_devices)
    datasets = {k: jnp.pad(v, ((0, pad_size), (0, 0))) for k, v in datasets.items()}

datasets_sharded = {
    k: jnp.reshape(v, (n_devices, -1, v.shape[-1])) 
    for k, v in datasets.items()
}
print(f"Sharded shape: {datasets_sharded['x'].shape}")  # (8, 1, 5) if 8 devices

# Create pmapped pipeline
# pmap the vmapped function: one per device
pcompute_sinusoid = pmap(
    vcompute_sinusoid,  # Re-use the vmapped version
    in_axes=(0, None),  # Shard the dataset axis 0, broadcast params
    axis_name="devices"
)

# Run on all devices
chi2_pmap = pcompute_sinusoid(datasets_sharded, params_sine)
print(f"PMAP χ² (sharded): {chi2_pmap}")  # shape: (8, 1)
print(f"PMAP χ² (flattened): {chi2_pmap.flatten()[:n_total]}")